# Second dataset: ScratchMath, reasoning arm only (Qwen2.5-VL-7B)

Fourteenth notebook. Every result in this project so far is on **one
dataset** (FERMAT). With three model families now agreeing on the
`has_error=1` stratified reasoning effect, a second dataset is the largest
remaining external-validity gap. This notebook is a **sizing + capability
run**, not a result: it measures whether the effect can be tested on
ScratchMath at all, before any full run is committed.

**Model held fixed at Qwen2.5-VL-7B** (confirmed 0.801 [0.751, 0.846] on
FERMAT at n=648) so the only thing varying is the dataset.

## What is different about ScratchMath, and why it constrains the design

1. **Every item contains an error.** There is no `has_error=0` stratum, so
   the clean-stratum inversion (confirmed on FERMAT at 0.280) *cannot be
   tested here*, and no balanced sample can be built. Only the
   `has_error=1` analysis is available -- which is, conveniently, exactly
   the cross-family-confirmed result.
2. **The question is a separate text field**; the image holds only the
   student's rough scratchwork, not a complete Question-Answer page. So the
   prompt supplies the question as text
   (`pilot.prompts.build_scratchmath_grading_prompt`), unlike FERMAT's
   single-image setup.
3. **The student's final answer is deliberately withheld from the model**,
   even though ScratchMath ships it as text. Handing it over would let the
   model check the arithmetic textually and reduce the image to decoration
   -- that would stop testing the vision-grounded claim this project makes.
   Locked by a test (`test_scratchmath.py`).
4. **Chinese questions, and much harder handwriting** than FERMAT -- sparse
   stylus/tablet scratchwork. A capability gate is a live risk.

## Pre-registered gate (fixed before this run)

All items are `has_error=1`, so the model's standing "there is an error"
bias scores well by construction and **grading accuracy is not the
quantity of interest** -- the misgrade count is, because that is the
minority class the AUROC needs.

```
misgrade rate >= 5%   -> FEASIBLE: >=30 misgrades reachable within
                         ScratchMath's 1720 items; proceed to extend.
misgrade rate <  5%   -> GATED: the model is answering "error" near-
                         universally and being right by construction;
                         report as a capability/degeneracy gate, do not
                         report an entropy number.
```
Also recorded, and checked before any number is interpreted: parse-failure
rate, says-error rate, and a qualitative read of whether the model's own
reasoning text engages with the image or is generic. **No AUROC is
reported from this n=100 run** -- 100 items cannot clear the registered
minimum of 30 misgrades under any plausible rate.


In [1]:
# Install cell: GPU-dependent packages only. Matches notebook 06 (Qwen path,
# so qwen-vl-utils is required; ScratchMath needs no extra dependency).
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 56.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 78.0 MB/s eta 0:00:00:00:0100:01


In [2]:
# Auth & code/results access cell. Identical to notebooks 06/09/10/11/12/13,
# including the sys.modules purge that makes a re-clone actually take effect
# (see notebook 13's comment for the bug that motivated it).
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.canonicalize
import pilot.plotting

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")
assert hasattr(pilot.data, "load_scratchmath_sample"), (
    "pilot.data has no load_scratchmath_sample -- the clone is stale. "
    "Push the ScratchMath loaders, then re-run this cell."
)
print("ScratchMath loaders present.")


Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.3 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot package imported from: /content/repo/pilot
ScratchMath loaders present.


In [3]:
# Model load cell. Qwen2.5-VL-7B, held fixed so the DATASET is the only
# thing varying vs the confirmed FERMAT result (0.801 [0.751, 0.846], n=648).
# Same loud 4-bit fallback as notebook 06 -- a quantized load is a different
# measurement, never a transparent substitute.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
QUANTIZED = False

try:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")

if QUANTIZED:
    print()
    print("WARNING: this run is 4-bit while the FERMAT comparison (0.801) is "
          "bf16. A dataset difference and a precision difference would be "
          "confounded. On a fresh runtime bf16 fits a 40 GiB A100 -- if this "
          "fired, restart the runtime before trusting any comparison.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-VL-7B-Instruct in bfloat16 (full precision).
GPU: NVIDIA A100-SXM4-40GB (39.5 GiB), quantized=False


In [4]:
# Sample cell. n=100 SIZING run over ScratchMath, all has_error=1 by
# construction (the dataset has no clean items -- see the notebook intro).
import logging

import pilot.data

logging.basicConfig(level=logging.INFO)

N = 100
SEED = 42

sample = pilot.data.load_scratchmath_sample(n=N, seed=SEED)
print(f"ScratchMath sample: {len(sample)} items")
print("study_level mix:",
      {lvl: sample["study_level"].count(lvl) for lvl in set(sample["study_level"])})

# Every ScratchMath item carries an error annotation. Assert it rather than
# assume it: if a future dataset revision added clean items, silently
# treating them as has_error=1 would corrupt every number downstream.
assert all(c is not None for c in sample["error_category"]), (
    "found an item with no error_category -- ScratchMath is expected to be "
    "all-error; re-check the dataset before proceeding"
)
print("Confirmed: all items are has_error=1 (no clean stratum exists here).")

_ex = sample[0]
print()
print("Example question:", str(_ex["question"])[:120])
print("Example image size:", _ex["student_scratchwork"].size)


README.md:   0%|          | 0.00/5.81k [00:00<?, ?B/s]

primary/data-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 89.7MB            

primary/data-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1479 [00:00<?, ? examples/s]

middle/data-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

middle/data-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/241 [00:00<?, ? examples/s]

ScratchMath sample: 100 items
study_level mix: {'primary': 86, 'middle': 14}
Confirmed: all items are has_error=1 (no clean stratum exists here).

Example question: 把一块长为80厘米，宽为60厘米的长方形的画框裱起来，四周都用金线装饰，金线每分米长度的价格是10元，那么装裱这个画框的四周需花费___1___元钱。
Example image size: (2880, 1638)


In [5]:
# Adapter + pre-flight. ScratchMath needs the question as TEXT alongside the
# scratchwork image, so this builds Qwen messages from the ScratchMath
# grading prompt rather than pilot.prompts.build_grading_messages (which
# assumes FERMAT's everything-in-one-image layout).
import pilot.prompts


def build_scratchmath_messages(item):
    """Qwen-shaped messages: ScratchMath system prompt + question text + image.

    The student's final answer is intentionally NOT passed -- see the intro
    and test_scratchmath.py::test_grading_prompt_never_leaks_the_students_final_answer.
    """
    user_prompt = pilot.prompts.build_scratchmath_grading_prompt(item["question"])
    return pilot.prompts.build_messages(
        pilot.prompts.SCRATCHMATH_GRADING_SYSTEM_PROMPT,
        user_prompt,
        item["student_scratchwork"],
    )


from qwen_vl_utils import process_vision_info

_test_messages = build_scratchmath_messages(sample[0])
_text_prompt = processor.apply_chat_template(
    _test_messages, tokenize=False, add_generation_prompt=True
)
_image_inputs, _video_inputs = process_vision_info(_test_messages)
_test_inputs = processor(
    text=[_text_prompt], images=_image_inputs, videos=_video_inputs,
    padding=True, return_tensors="pt",
).to(model.device)

with torch.no_grad():
    _test_output = model.generate(**_test_inputs, max_new_tokens=256, do_sample=False)
_test_text = processor.batch_decode(
    _test_output[:, _test_inputs["input_ids"].shape[1]:], skip_special_tokens=True
)[0]

print("Pre-flight OK. Greedy sample output:")
print(_test_text)
assert len(_test_text.strip()) > 0, "empty output -- adapter is broken"
assert pilot.parsing.parse_grading(_test_text) is not None, (
    "pre-flight output did not parse to a 0/1 digit -- check the prompt's "
    "**Error:** contract before spending the sampling budget"
)


Pre-flight OK. Greedy sample output:
**Reasoning:** The image shows two numbers, 140 and 286, but there is no indication of a calculation being performed. The question asks for the cost of decorating the perimeter of a rectangular frame with gold thread, which requires calculating the perimeter first. The given numbers do not seem to be related to the problem at hand.

**Error:** 1


In [6]:
# Grading generation, K=5. Same batch-backoff ladder and checkpoint/resume
# structure as notebooks 06/13, pointed at ScratchMath items.
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("question_id", "question", "answer", "student_answer",
               "error_category", "study_level")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_grading(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
checkpoint_path = (f"{CHECKPOINT_DIR}/scratchmath_{model_slug}_n{N}_seed{SEED}"
                   f"_kg{K_GRADING}{'_4bit' if QUANTIZED else ''}.jsonl")

raw_results = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        raw_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(raw_results[:N]):
        item = sample[idx]
        if entry["item"].get("question_id") != item["question_id"]:
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if len(entry.get("grading_samples_raw", [])) != K_GRADING:
            break
        valid.append(entry)
    if len(valid) != len(raw_results):
        with open(checkpoint_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
        print(f"Truncated checkpoint from {len(raw_results)} to {len(valid)} valid items.")
    raw_results = valid
    print(f"Resuming from {len(raw_results)} completed items")

if len(raw_results) >= N:
    print(f"All {N} items already done.")
else:
    print(f"Starting from item {len(raw_results) + 1}/{N}", flush=True)
    with tqdm(total=(N - len(raw_results)) * K_GRADING, desc="grading", unit="sample") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < len(raw_results):
                continue
            _t0 = time.time()
            texts = generate_grading(build_scratchmath_messages(item), K_GRADING, TEMP)
            pbar.update(K_GRADING)
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "grading_samples_raw": texts,
                "quantized": QUANTIZED,
                "elapsed_seconds": time.time() - _t0,
            }
            raw_results.append(entry)
            with open(checkpoint_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            print(f"  item {item_idx + 1}/{N}: {time.time() - _t0:.1f}s", flush=True)

print(f"raw_results: {len(raw_results)} items")


Starting from item 1/100


grading:   0%|          | 0/500 [00:00<?, ?sample/s]

  item 1/100: 14.4s
  item 2/100: 9.2s
  item 3/100: 8.3s
  item 4/100: 9.5s
  item 5/100: 10.5s
  item 6/100: 9.9s
  item 7/100: 11.9s
  item 8/100: 8.4s
  item 9/100: 12.0s
  item 10/100: 12.3s
  item 11/100: 10.8s
  item 12/100: 7.4s
  item 13/100: 7.9s
  item 14/100: 8.8s
  item 15/100: 10.1s
  item 16/100: 9.0s
  item 17/100: 8.9s
  item 18/100: 9.9s
  item 19/100: 14.8s
  item 20/100: 15.7s
  item 21/100: 9.6s
  item 22/100: 10.3s
  item 23/100: 8.5s
  item 24/100: 7.6s
  item 25/100: 13.7s
  item 26/100: 11.3s
  item 27/100: 9.6s
  item 28/100: 14.1s
  item 29/100: 11.4s
  item 30/100: 10.2s
  item 31/100: 7.8s
  item 32/100: 15.5s
  item 33/100: 10.5s
  item 34/100: 9.0s
  item 35/100: 7.8s
  item 36/100: 10.4s
  item 37/100: 12.5s
  item 38/100: 11.5s
  item 39/100: 10.7s
  item 40/100: 9.8s
  item 41/100: 10.4s
  item 42/100: 8.3s
  item 43/100: 9.9s
  item 44/100: 8.7s
  item 45/100: 7.8s
  item 46/100: 7.4s
  item 47/100: 11.2s
  item 48/100: 10.9s
  item 49/100: 11.4s
  it

In [7]:
# Scoring + the pre-registered gate. Deliberately reports NO AUROC: at
# n=100, no plausible misgrade rate reaches the registered minimum of 30,
# so an AUROC here would be an uninterpretable number that invites being
# quoted. This cell answers one question only -- is the full run feasible?
import pandas as pd

import pilot.entropy
import pilot.parsing

GATE_MIN_MISGRADE_RATE = 0.05  # pre-registered, see the intro markdown
REGISTERED_MIN_MINORITY = pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]

rows = []
for entry in raw_results:
    item = entry["item"]
    samples = entry["grading_samples_raw"]
    parsed = [pilot.parsing.parse_grading(s) for s in samples]
    labels = [None if d is None else str(d) for d in parsed]
    majority, _ = pilot.entropy.majority_cluster(labels)
    digits = [d for d in parsed if d is not None]
    rows.append({
        "question_id": item["question_id"],
        "question": item["question"],
        "answer": item["answer"],
        "student_answer": item["student_answer"],
        "error_category": item["error_category"],
        "study_level": item["study_level"],
        "has_error": 1,  # every ScratchMath item, by construction
        "reasoning_entropy": pilot.entropy.cluster_entropy(labels),
        "grading_correct": majority == "1",
        "n_grading_parse_failures": sum(1 for d in parsed if d is None),
        "says_error_frac": (sum(digits) / len(digits)) if digits else float("nan"),
        "all_grading_samples_raw": samples,
        "model_id": MODEL_ID,
        "quantized": QUANTIZED,
    })

df = pd.DataFrame(rows)

n_misgraded = int((~df["grading_correct"]).sum())
misgrade_rate = n_misgraded / len(df)
parse_fail_rate = df["n_grading_parse_failures"].sum() / (len(df) * K_GRADING)

print(f"n                 : {len(df)}")
print(f"grading accuracy  : {df['grading_correct'].mean():.1%}  "
      f"(NOT the quantity of interest -- all items are has_error=1, so the "
      f"model's error-bias scores well by construction)")
print(f"says-error rate   : {df['says_error_frac'].mean():.1%}")
print(f"parse failures    : {parse_fail_rate:.1%}")
print(f"MISGRADED         : {n_misgraded}/{len(df)} = {misgrade_rate:.1%}   <- the minority class")
print()

if misgrade_rate >= GATE_MIN_MISGRADE_RATE:
    needed = int(REGISTERED_MIN_MINORITY / misgrade_rate)
    print(f"GATE: FEASIBLE (misgrade rate {misgrade_rate:.1%} >= {GATE_MIN_MISGRADE_RATE:.0%}).")
    print(f"  Projected total items for {REGISTERED_MIN_MINORITY} misgrades: ~{needed} "
          f"(ScratchMath has 1720, so this is reachable).")
    print(f"  Next step: extend with pilot.data.load_scratchmath_extra("
          f"n_extra=~{max(0, needed - len(df))}, seed={SEED}, skip={len(df)}).")
else:
    print(f"GATE: GATED (misgrade rate {misgrade_rate:.1%} < {GATE_MIN_MISGRADE_RATE:.0%}).")
    print("  The model is answering 'error' near-universally and is right by "
          "construction on an all-error dataset. Report this as a "
          "capability/degeneracy gate -- do NOT report an entropy AUROC.")

print()
print("Qualitative check -- read these before trusting the gate verdict. Does the "
      "model engage with the IMAGE, or is the reasoning generic boilerplate that "
      "would read the same for any scratchwork?")
for i in range(min(3, len(df))):
    print("=" * 70)
    print("Q:", str(df.iloc[i]["question"])[:100])
    print("sample 0:", str(df.iloc[i]["all_grading_samples_raw"][0])[:400])


n                 : 100
grading accuracy  : 96.0%  (NOT the quantity of interest -- all items are has_error=1, so the model's error-bias scores well by construction)
says-error rate   : 90.0%
parse failures    : 0.0%
MISGRADED         : 4/100 = 4.0%   <- the minority class

GATE: GATED (misgrade rate 4.0% < 5%).
  The model is answering 'error' near-universally and is right by construction on an all-error dataset. Report this as a capability/degeneracy gate -- do NOT report an entropy AUROC.

Qualitative check -- read these before trusting the gate verdict. Does the model engage with the IMAGE, or is the reasoning generic boilerplate that would read the same for any scratchwork?
Q: 把一块长为80厘米，宽为60厘米的长方形的画框裱起来，四周都用金线装饰，金线每分米长度的价格是10元，那么装裱这个画框的四周需花费___1___元钱。
sample 0: **Reasoning:** The image only shows two numbers: 140 and 286. There is no indication of any calculation or application related to the question about the cost of decorating a rectangular frame with gold wire. The numbers do 

In [8]:
# Save cell: Drive first, then repo + push. Distinct filename -- never
# collides with any FERMAT results file.
import os
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
model_slug_lower = MODEL_ID.split("/")[-1].lower().replace(".", "")
csv_name = (f"scratchmath_sizing_n{N}_"
            f"{'4bit_' if QUANTIZED else ''}{model_slug_lower}_{timestamp}.csv")

drive_results = f"{PROJECT_DIR}/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add ScratchMath sizing-run results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")


Backup written to /content/drive/MyDrive/uncertainty-math-vlm/results/scratchmath_sizing_n100_qwen25-vl-7b-instruct_20260808T121016Z.csv
Wrote repo/results/scratchmath_sizing_n100_qwen25-vl-7b-instruct_20260808T121016Z.csv (100 rows)
remote: Permission to sepehrmaleki369/uncertainty-math-vlm.git denied to sepehrmaleki369.
fatal: unable to access 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git/': The requested URL returned error: 403
Push failed (see above). The CSV is safe on Drive and in repo/results/ -- retry the push without re-running the model.
